In [3]:
from pathlib import Path
import os

print("CWD:", os.getcwd())

RAW_BASE = Path("../../data/raw").resolve()   # <-- FIX: go up 2 levels
print("RAW_BASE:", RAW_BASE)
print("Exists:", RAW_BASE.exists())

print("\nTop-level folders inside data/raw:\n")
for p in RAW_BASE.iterdir():
    print("-", p.name)


CWD: c:\Users\User\Desktop\Vehicle_Damage_Detection\notebooks\02_preprocess
RAW_BASE: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw
Exists: True

Top-level folders inside data/raw:

- archive (1)
- archive (2)
- archive (3)
- archive (4)
- archive (5)
- CarDD_release


In [4]:
CARDD = RAW_BASE / "CarDD_release"
print("CARDD:", CARDD)
print("Exists:", CARDD.exists())

print("\nContents of CarDD_release:\n")
for p in CARDD.iterdir():
    print("-", p.name)


CARDD: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release
Exists: True

Contents of CarDD_release:

- CarDD_release


In [5]:
CARDD_ROOT = RAW_BASE / "CarDD_release" / "CarDD_release"

print("CARDD_ROOT:", CARDD_ROOT)
print("Exists:", CARDD_ROOT.exists())

print("\nContents of CARDD_ROOT:\n")
for p in CARDD_ROOT.iterdir():
    print("-", p.name)


CARDD_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release
Exists: True

Contents of CARDD_ROOT:

- CarDD_COCO
- CarDD_SOD


In [6]:
CARDD_COCO = CARDD_ROOT / "CarDD_COCO"

print("CARDD_COCO:", CARDD_COCO)
print("Exists:", CARDD_COCO.exists())

print("\nContents of CarDD_COCO:\n")
for p in CARDD_COCO.iterdir():
    print("-", p.name)


CARDD_COCO: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_COCO
Exists: True

Contents of CarDD_COCO:

- annotations
- test2017
- train2017
- val2017


In [12]:
from pathlib import Path
import shutil

# -----------------------------
# AUTO paths detection
# -----------------------------
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Vehicle_Damage_Detection" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_BASE = PROJECT_ROOT / "data" / "raw"
CARDD_ROOT = RAW_BASE / "CarDD_release" / "CarDD_release"
CARDD_COCO = CARDD_ROOT / "CarDD_COCO"

OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "cardd_coco"
OUT_ANN  = OUT_ROOT / "annotations"

# -----------------------------
# Checks
# -----------------------------
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_BASE exists:", RAW_BASE.exists())
print("CARDD_COCO exists:", CARDD_COCO.exists())

splits = ["train2017", "val2017", "test2017"]
ann_files = [
    "instances_train2017.json",
    "instances_val2017.json",
    "instances_test2017.json",
]

# Make output folders
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ANN.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Copy images (folders)
# -----------------------------
total_copied_imgs = 0
for s in splits:
    src_dir = CARDD_COCO / s
    dst_dir = OUT_ROOT / s
    print(f"\n[{s}] src exists:", src_dir.exists())

    dst_dir.mkdir(parents=True, exist_ok=True)

    # copy files (skip if already copied)
    copied = 0
    for img in src_dir.glob("*"):
        if img.is_file():
            out_f = dst_dir / img.name
            if not out_f.exists():
                shutil.copy2(img, out_f)
                copied += 1
    total_copied_imgs += copied
    print(f"[{s}] newly copied:", copied)

# -----------------------------
# Copy annotation jsons
# -----------------------------
print("\nCopying annotations...")
for f in ann_files:
    src = CARDD_COCO / "annotations" / f
    dst = OUT_ANN / f
    print("-", f, "| exists:", src.exists())
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

print("\n.. PREPROCESS DONE")
print("Saved to:", OUT_ROOT)
print("Newly copied images (this run):", total_copied_imgs)


PROJECT_ROOT: c:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_BASE exists: True
CARDD_COCO exists: True

[train2017] src exists: True
[train2017] newly copied: 41

[val2017] src exists: True
[val2017] newly copied: 810

[test2017] src exists: True
[test2017] newly copied: 374

Copying annotations...
- instances_train2017.json | exists: True
- instances_val2017.json | exists: True
- instances_test2017.json | exists: True

.. PREPROCESS DONE
Saved to: c:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco
Newly copied images (this run): 1225


In [10]:
from pathlib import Path

out = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco")

print("train2017:", len(list((out/"train2017").glob("*"))))
print("val2017  :", len(list((out/"val2017").glob("*"))))
print("test2017 :", len(list((out/"test2017").glob("*"))))
print("ann train:", (out/"annotations"/"instances_train2017.json").exists())


train2017: 2816
val2017  : 810
test2017 : 374
ann train: True


In [1]:
import json
from pathlib import Path
from collections import Counter

# locate processed folder
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Vehicle_Damage_Detection" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

P = PROJECT_ROOT / "data" / "processed" / "cardd_coco"

print("VERIFY:", P)
print("-" * 60)

# image counts
for s in ["train2017", "val2017", "test2017"]:
    n = len([x for x in (P / s).glob("*") if x.is_file()])
    print(f"{s:8} images:", n)

# annotation + classes
ann_path = P / "annotations" / "instances_train2017.json"
print("\nTrain annotation exists:", ann_path.exists())

if ann_path.exists():
    coco = json.load(open(ann_path, "r", encoding="utf-8"))
    id_to_name = {c["id"]: c["name"] for c in coco.get("categories", [])}
    counts = Counter(a["category_id"] for a in coco.get("annotations", []))

    print("\nClasses found:", len(id_to_name))
    print("Annotations (boxes) total:", sum(counts.values()))
    print("\nBoxes per class:")
    for cid, name in sorted(id_to_name.items()):
        print(f"- {name:15} : {counts.get(cid, 0)}")


VERIFY: c:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_coco
------------------------------------------------------------
train2017 images: 2816
val2017  images: 810
test2017 images: 374

Train annotation exists: True

Classes found: 6
Annotations (boxes) total: 6211

Boxes per class:
- dent            : 1806
- scratch         : 2560
- crack           : 651
- glass shatter   : 475
- lamp broken     : 494
- tire flat       : 225
